# SI/MA neural coding: Glutamatergic vs. GABAergic groups

This notebook reproduces the polar / diagonal significance analysis of the
reference call

```python
run_model_plot(
    ds=ds,
    plot_methods=["polar", "diag"],
    models=["qlearn_reward_vs_chosenQ_g0"],
    regions=[["SI", "MA"]],
    time_window="0.3_2_-1_0",
    ...
)
```

but **splits the SI/MA (ventral pallidum) units into two putative cell-type
groups** using a simple waveform-feature threshold, then plots the polar figure
**separately** for each group so the neural coding can be compared.

**Strategy**
1. Load the per-unit correlation results (`ds`).
2. Load the waveform-feature table and select the SI/MA units.
3. Split those units by a single waveform feature (default:
   `repolarization_slope`): feature `> GLUT_SLOPE_MIN` → **Glutamatergic**,
   feature `< GABA_SLOPE_MAX` → **GABAergic**, in-between → unassigned.
4. Map those group labels back onto `ds` (via `session_name` + `unit_index`),
   split `ds`, and draw the polar plot for each group.


## 1. Environment setup & imports

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
import re
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from general_utils import load_temporary_data
from waveform_clustering import (
    FEATURE_COLS,
    load_features_dataset,
    normalize_waveform,
    clean_waveform_mask,
)
from ephys_behavior_visualization import (
    plot_angle_fraction_polar,
    plot_diagonal_significance,
)

%matplotlib inline
print("Imports ready.")


## 2. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# --- Correlation results (per-unit regression coefficients / p-values) -------
ZARR_PATH = "/root/capsule/scratch/correlation_results/sig_dir_all_sessions.zarr"

# --- Waveform features -------------------------------------------------------
DATASET_CSV = Path("/root/capsule/scratch/waveform_clustering/all_sessions_waveform_features.csv")

# --- Region + analysis window ------------------------------------------------
TARGET_REGIONS = ["SI", "MA"]          # ventral pallidum
TIME_WINDOW = "0.3_2_-1_0"             # same window as the reference polar plot

# --- Model columns (qlearn_reward_vs_chosenQ_g0) -----------------------------
MODEL_COLS = {
    "col_x": "simple_LR-QLearning_L2F1_softmax-reward-g0-s0-d0-coef",
    "col_y": "simple_LR-QLearning_L2F1_softmax-chosenQ-g0-s0-d0-coef",
    "col_pval_x": "simple_LR-QLearning_L2F1_softmax-reward-g0-s0-d0-pval",
    "col_pval_y": "simple_LR-QLearning_L2F1_softmax-chosenQ-g0-s0-d0-pval",
}

# --- Polar plot options (match the reference call) ---------------------------
POLAR_KW = dict(
    include="all",
    angle_bin_deg=15,
    normalize="selected",
    start_angle_deg=0,
    show_reference_diagonals=True,
)

# --- Waveform-based grouping -------------------------------------------------
# Split the SI/MA units into two putative cell-type groups using a single
# waveform feature and two thresholds. Units that fall between the thresholds
# are left "Unassigned" and excluded from both group plots.
SPLIT_FEATURE = "repolarization_slope"
GLUT_SLOPE_MIN = 8.0     # feature >  GLUT_SLOPE_MIN  -> Glutamatergic
GABA_SLOPE_MAX = 5.0     # feature <  GABA_SLOPE_MAX  -> GABAergic

# --- Noise filtering for waveforms -------------------------------------------
FILTER_NOISE = True
NOISE_MAX_PEAKS = 4
NOISE_MAX_ZERO_CROSSINGS = 8
NOISE_MAX_LATE_ENERGY = 0.6
NOISE_CORE_MS = 0.7

print("Config set. Window:", TIME_WINDOW, "| Regions:", TARGET_REGIONS)
print(f"Split: {SPLIT_FEATURE} > {GLUT_SLOPE_MIN} -> Glut | < {GABA_SLOPE_MAX} -> GABA")


## 3. Load the correlation results (`ds`)

In [ ]:
# One row per unit per time window (long form). Units are keyed by
# (session_name, unit_index); the region column is `brain_region`.
ds = load_temporary_data(ZARR_PATH)
print("ds rows:", len(ds))
print("Key columns present:",
      [c for c in ["brain_region", "time_window", "session_name", "unit_index"] if c in ds.columns])
ds.head()

## 4. Load the waveform features and build the SI/MA population mask

In [ ]:
# Reload the flat waveform/feature CSV into a feature table + waveform matrix.
loaded = load_features_dataset(DATASET_CSV)
features = loaded["features"]
waveforms = loaded["waveforms"]
time_ms = loaded["time_ms"]

# Amplitude-normalize each stored waveform (used by the noise filter + plots).
norm_waveforms = (
    np.vstack([normalize_waveform(w) for w in waveforms]) if len(waveforms) else waveforms
)

# QC mask.
qc_mask = features["qc_pass"].astype(bool).values

# Noise / artifact filter (drop oscillatory, non-spike waveforms).
if FILTER_NOISE:
    _cm = clean_waveform_mask(
        norm_waveforms, time_ms,
        max_peaks=NOISE_MAX_PEAKS,
        max_zero_crossings=NOISE_MAX_ZERO_CROSSINGS,
        max_late_energy=NOISE_MAX_LATE_ENERGY,
        core_ms=NOISE_CORE_MS,
    )
    clean_mask = np.asarray(_cm[0] if isinstance(_cm, tuple) else _cm, dtype=bool)
else:
    clean_mask = np.ones(len(features), dtype=bool)

# SI/MA population (QC + non-noise).
region_mask = features["region"].isin(TARGET_REGIONS).values & qc_mask & clean_mask
print(f"Total units: {len(features)} | QC-pass: {int(qc_mask.sum())} | non-noise: {int(clean_mask.sum())}")
print(f"SI/MA units (QC + non-noise): {int(region_mask.sum())}")
print(features.loc[region_mask, "region"].value_counts())

## 6. Split SI/MA into two waveform groups by threshold

Assign each SI/MA unit to a group using `SPLIT_FEATURE` (default
`repolarization_slope`): feature `> GLUT_SLOPE_MIN` → **Glutamatergic**,
feature `< GABA_SLOPE_MAX` → **GABAergic**. Units between the two thresholds are
left **Unassigned** and dropped from the group comparison.


In [ ]:
sima_feat = features.loc[region_mask].copy().reset_index(drop=True)

# Threshold-based split on a single waveform feature.
vals = sima_feat[SPLIT_FEATURE]
sima_feat["wf_group"] = np.where(
    vals > GLUT_SLOPE_MIN, "Glutamatergic",
    np.where(vals < GABA_SLOPE_MAX, "GABAergic", "Unassigned"),
)

print(f"Split feature: {SPLIT_FEATURE}")
print(f"  Glutamatergic ({SPLIT_FEATURE} > {GLUT_SLOPE_MIN})")
print(f"  GABAergic     ({SPLIT_FEATURE} < {GABA_SLOPE_MAX})")
print(f"  Unassigned    ({GABA_SLOPE_MAX} <= {SPLIT_FEATURE} <= {GLUT_SLOPE_MIN})")
print(sima_feat["wf_group"].value_counts())

# Per-group feature means (sanity check of the split).
sima_feat.groupby("wf_group")[FEATURE_COLS].mean()


In [ ]:
# Visualize the split: feature-space scatter + the split-feature distribution.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {"Glutamatergic": "tab:blue", "GABAergic": "tab:orange", "Unassigned": "lightgrey"}

# (a) morphology feature space, coloured by group.
ax = axes[0]
for g, c in colors.items():
    m = (sima_feat["wf_group"] == g).to_numpy()
    if not m.any():
        continue
    ax.scatter(sima_feat.loc[m, "trough_to_peak_ms"],
               sima_feat.loc[m, "half_width_ms"],
               s=25, c=c, alpha=0.6, edgecolor="none",
               label=f"{g} (n={int(m.sum())})")
ax.set_xlabel("Trough-to-peak duration (ms)")
ax.set_ylabel("Half-width (ms)")
ax.set_title("SI/MA units by waveform group")
ax.legend()

# (b) split-feature histogram with the two thresholds.
ax = axes[1]
ax.hist(sima_feat[SPLIT_FEATURE].dropna(), bins=40, color="grey", alpha=0.8)
ax.axvline(GABA_SLOPE_MAX, color="tab:orange", linestyle="--",
           label=f"GABAergic < {GABA_SLOPE_MAX}")
ax.axvline(GLUT_SLOPE_MIN, color="tab:blue", linestyle="--",
           label=f"Glutamatergic > {GLUT_SLOPE_MIN}")
ax.set_xlabel(SPLIT_FEATURE)
ax.set_ylabel("Count")
ax.set_title(f"{SPLIT_FEATURE} distribution (SI/MA)")
ax.legend()

plt.tight_layout()
plt.show()


## 7. Map the group labels onto `ds` and split it

`plot_angle_fraction_polar` filters `ds` only by region + time window, so we
pre-split `ds` into a Glutamatergic subset and a GABAergic subset using the
`(session_name, unit_index)` group map.

In [ ]:
group_map = {
    f"{s}|{int(u)}": g
    for s, u, g in zip(sima_feat["session_name"], sima_feat["unit_index"], sima_feat["wf_group"])
}

ds_sima = ds[(ds["brain_region"].isin(TARGET_REGIONS)) & (ds["time_window"] == TIME_WINDOW)].copy()
ds_key = ds_sima["session_name"].astype(str) + "|" + ds_sima["unit_index"].astype(int).astype(str)
ds_sima["wf_group"] = ds_key.map(group_map)

n_unmapped = int(ds_sima["wf_group"].isna().sum())
print(f"SI/MA rows in ds for window {TIME_WINDOW}: {len(ds_sima)}")
print(f"  mapped to a waveform group: {len(ds_sima) - n_unmapped}")
print(f"  unmapped (no matching waveform unit): {n_unmapped}")
print(ds_sima["wf_group"].value_counts(dropna=False))

ds_glut = ds_sima[ds_sima["wf_group"] == "Glutamatergic"].copy()
ds_gaba = ds_sima[ds_sima["wf_group"] == "GABAergic"].copy()

## 8. Polar comparison: Glutamatergic vs. GABAergic

Same model (`reward` vs `chosenQ`, g0) and window as the reference call, but
drawn separately for each group.

In [ ]:
def polar_for_group(ds_group, label):
    """Draw the reward-vs-chosenQ polar plot for one waveform group."""
    if len(ds_group) == 0:
        print(f"[skip] {label}: no units.")
        return None
    fig, ax, tbl = plot_angle_fraction_polar(
        ds=ds_group,
        filter_region=TARGET_REGIONS,
        time_window=TIME_WINDOW,
        **MODEL_COLS,
        **POLAR_KW,
    )
    ax.set_title(f"{label} SI/MA (n={len(ds_group)})\nreward vs chosenQ (g0), {TIME_WINDOW}")
    plt.show()
    return fig, ax, tbl


res_glut = polar_for_group(ds_glut, "Glutamatergic")
res_gaba = polar_for_group(ds_gaba, "GABAergic")

### 8a. (Optional) Diagonal-significance view per group

In [ ]:
def diag_for_group(ds_group, label):
    """Draw the diagonal-significance scatter for one waveform group."""
    if len(ds_group) == 0:
        print(f"[skip] {label}: no units.")
        return None
    out = plot_diagonal_significance(
        ds=ds_group,
        filter_region=TARGET_REGIONS,
        time_window=TIME_WINDOW,
        **MODEL_COLS,
        point_size=1,
        fit_oval=False,
    )
    if isinstance(out, tuple) and len(out) >= 2:
        fig, ax = out[:2]
        ax.set_title(f"{label} SI/MA (n={len(ds_group)}) - reward vs chosenQ (g0), {TIME_WINDOW}")
    plt.show()
    return out


diag_for_group(ds_glut, "Glutamatergic")
diag_for_group(ds_gaba, "GABAergic")

## 9. Notes

- The two groups are defined **purely by a threshold on a single waveform
  feature** (`SPLIT_FEATURE`, default `repolarization_slope`): `> GLUT_SLOPE_MIN`
  → Glutamatergic, `< GABA_SLOPE_MAX` → GABAergic.
- Units between the two thresholds are left **Unassigned** and excluded from the
  group plots. Set `GLUT_SLOPE_MIN == GABA_SLOPE_MAX` for a hard split with no
  unassigned band, or change `SPLIT_FEATURE` to another column in `FEATURE_COLS`.
- Unmapped `ds` rows are SI/MA correlation units that have no matching entry in
  the waveform-feature table (their session/unit is missing, failed QC, or was
  flagged as noise). They are excluded from both group plots.
